In [ ]:
-- Unicite

SELECT u.*
FROM read_parquet('/data/data_quality/unicite.parquet') u
JOIN (
  SELECT source, column_name, MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/unicite.parquet')
  GROUP BY source, column_name
) d
  ON u.source = d.source
 AND u.column_name = d.column_name
 AND u.timestamp = d.dernier;

In [ ]:
-- coherence

SELECT c.*
FROM read_parquet('/data/data_quality/coherence.parquet') c
JOIN (
  SELECT source, x, y, MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/coherence.parquet')
  GROUP BY source, x, y
) d
  ON c.source = d.source
 AND c.x = d.x
 AND c.y = d.y
 AND c.timestamp = d.dernier;

In [ ]:
-- completude

SELECT c.*
FROM read_parquet('/data/data_quality/completude.parquet') c
JOIN (
  SELECT source, column_name, completude_percent, MAX(TIMESTAMP) as dernier
  FROM read_parquet('/data/data_quality/completude.parquet')
  GROUP BY source, column_name, completude_percent
  ) d 
ON  c.source = d.source
AND c.column_name = d.column_name
AND c.completude_percent = d.completude_percent
AND c.timestamp = d.dernier;

In [ ]:
-- conformite format

SELECT t.*
FROM read_parquet('/data/data_quality/conformite_fmt.parquet') t
JOIN (
  SELECT source, column_name, MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/conformite_fmt.parquet')
  GROUP BY source, column_name
) d
  ON t.source = d.source
 AND t.column_name = d.column_name
 AND t.timestamp = d.dernier;

In [ ]:
--conformite label

SELECT t.*
FROM read_parquet('/data/data_quality/conformite_label.parquet') t
JOIN (
  SELECT source, column_name, MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/conformite_label.parquet')
  GROUP BY source, column_name
) d
  ON t.source = d.source
 AND t.column_name = d.column_name
 AND t.timestamp = d.dernier;

In [ ]:
--conformite intervalle

SELECT t.*
FROM read_parquet('/data/data_quality/conformite_interval.parquet') t
JOIN (
  SELECT source, column_name_min, column_name_target, column_name_max,
         MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/conformite_interval.parquet')
  GROUP BY source, column_name_min, column_name_target, column_name_max
) d
  ON t.source = d.source
 AND t.column_name_min = d.column_name_min
 AND t.column_name_target = d.column_name_target
 AND t.column_name_max = d.column_name_max
 AND t.timestamp = d.dernier;

In [ ]:
-- conformite clé etrangère

SELECT t.*
FROM read_parquet('/data/data_quality/conformite_fk.parquet') t
JOIN (
  SELECT source_1, column_name_1, source_2, column_name_2, MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/conformite_fk.parquet')
  GROUP BY source_1, column_name_1, source_2, column_name_2
) d
  ON t.source_1 = d.source_1
 AND t.column_name_1 = d.column_name_1
 AND t.source_2 = d.source_2
 AND t.column_name_2 = d.column_name_2
 AND t.timestamp = d.dernier;

In [ ]:
-- heterogeneite dispersion

SELECT t.*
FROM read_parquet('/data/data_quality/heterogeneite_dispersion.parquet') t
JOIN (
  SELECT source_1, column_name_1, source_2, column_name_2, MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/heterogeneite_dispersion.parquet')
  GROUP BY source_1, column_name_1, source_2, column_name_2
) d
  ON t.source_1 = d.source_1
 AND t.column_name_1 = d.column_name_1
 AND t.source_2 = d.source_2
 AND t.column_name_2 = d.column_name_2
 AND t.timestamp = d.dernier;

In [ ]:
-- heterogeneite ratio

SELECT t.*
FROM read_parquet('/data/data_quality/heterogeneite_ratio.parquet') t
JOIN (
  SELECT source, column_name_min, column_name_mean, column_name_max,
         MAX(timestamp) AS dernier
  FROM read_parquet('/data/data_quality/heterogeneite_ratio.parquet')
  GROUP BY source, column_name_min, column_name_mean, column_name_max
) d
  ON t.source = d.source
 AND t.column_name_min = d.column_name_min
 AND t.column_name_mean = d.column_name_mean
 AND t.column_name_max = d.column_name_max
 AND t.timestamp = d.dernier;

In [ ]:
WITH toutes AS (
  SELECT 'Complétude' AS dimension, 'Valeurs non nulles' AS metrique,
         source, column_name AS regle, completude_percent AS score, timestamp
  FROM read_parquet('/data/data_quality/completude.parquet')

  UNION ALL
  SELECT 'Unicité', 'Absence de doublons',
         source, column_name, unicite_percent, timestamp
  FROM read_parquet('/data/data_quality/unicite.parquet')

  UNION ALL
  SELECT 'Conformité', 'Format',
         source, column_name, format_percent, timestamp
  FROM read_parquet('/data/data_quality/conformite_fmt.parquet')

  UNION ALL
  SELECT 'Conformité', 'Valeurs autorisées',
         source, column_name, labels_percent, timestamp
  FROM read_parquet('/data/data_quality/conformite_label.parquet')

  UNION ALL
  SELECT 'Conformité', 'Intervalle',
         source, column_name_min || ' ≤ ' || column_name_target || ' ≤ ' || column_name_max,
         1 - interval_percent, timestamp
  FROM read_parquet('/data/data_quality/conformite_interval.parquet')

  UNION ALL
  SELECT 'Conformité', 'Clé étrangère',
         source_1 || ' -> ' || source_2, column_name_1 || ' -> ' || column_name_2,
         fk_percent, timestamp
  FROM read_parquet('/data/data_quality/conformite_fk.parquet')

  UNION ALL
  SELECT 'Hétérogénéité', 'Dispersion entre sources',
         source_1 || ' vs ' || source_2, column_name_1,
         GREATEST(0, 1 - difference_dispersion), timestamp
  FROM read_parquet('/data/data_quality/heterogeneite_dispersion.parquet')

  UNION ALL
  SELECT 'Hétérogénéité', 'Position de la moyenne',
         source, column_name_min || ' / ' || column_name_mean || ' / ' || column_name_max,
         1 - ratio_position, timestamp
  FROM read_parquet('/data/data_quality/heterogeneite_ratio.parquet')
),
dernieres AS (
  SELECT dimension, metrique, source, regle, MAX(timestamp) AS dernier
  FROM toutes
  GROUP BY dimension, metrique, source, regle
)
SELECT t.*,
       t.timestamp = d.dernier AS derniere_mesure,
       CASE
         WHEN t.source IN ('population1', 'consommation1') THEN 'Paris'
         WHEN t.source IN ('population2', 'consommation2') THEN 'Évry'
         ELSE 'Commun'
       END AS ville
FROM toutes t
JOIN dernieres d
  ON t.dimension = d.dimension
 AND t.metrique = d.metrique
 AND t.source = d.source
 AND t.regle = d.regle;